## Pyspark 

O que é ?
é a biblioteca do Python que permite usar o Apache Spark, um poderoso motor de computação em cluster para processamento de dados em larga escala. Ele serve para analisar, transformar e manipular Big Data de forma extremamente rápida através de processamento distribuído.

In [30]:
# Iniciando Sessão 
from pyspark.sql import SparkSession

Criando uma sessão

In [31]:
session = SparkSession \
    .builder\
    .appName(name='Estudando Pyspark') \
    .getOrCreate()

> `remote` : defini a url da sessão
> `appName` : Nome da aplicação
> `getOrCreate` : Criação da aplicação

In [32]:
session.stop()

Funcionalidade de .config na construção da sessão

#### Configurações de Aplicativo e Recursos Básicos
Controlam o nome do app, onde ele roda e quantos recursos (CPU/Memória) ele vai usar.

* spark.app.name: O nome do seu aplicativo (visível na Spark UI).
* spark.master: Onde o Spark vai rodar (ex: local[*], yarn, k8s://..., spark://...).
* spark.executor.memory: Quantidade de memória por executor (ex: 4g, 16g).
* spark.driver.memory: Memória para o processo Driver (onde o script Python roda).
* spark.executor.cores: Número de cores (threads) por executor.
* spark.cores.max: Número máximo de cores para o aplicativo inteiro (em cluster).

####  Gerenciamento e Alocação de Memória
Essas são cruciais para evitar os famosos erros de OutOfMemory (OOM).

* spark.memory.fraction: Fração do espaço de memória de execução e armazenamento (padrão é 0.6).
* spark.memory.storageFraction: Quanto da memória acima será protegida para cache/persistência (padrão 0.5).
* spark.executor.memoryOverhead: Memória off-heap adicional para processos do sistema/JVM por executor.
* spark.driver.memoryOverhead: Memória off-heap adicional para o Driver.
* spark.python.worker.memory: Limite de memória para os workers Python do PySpark.


#### Shuffle e Performance (Spark SQL / DataFrames)
Controlam como o Spark move dados entre os nós e processa consultas.

* spark.sql.shuffle.partitions: Número de partições ao fazer um shuffle (joins, agregações). O padrão é 200 (geralmente alto para dados locais, baixo para Big Data).
* spark.sql.adaptive.enabled: Ativa o Adaptive Query Execution (AQE), que otimiza o plano de execução em tempo real (padrão true).
* spark.sql.adaptive.coalescePartitions.enabled: Permite ao AQE juntar partições pequenas automaticamente.
* spark.sql.autoBroadcastJoinThreshold: Tamanho máximo (em bytes) de uma tabela para sofrer um Broadcast Join (evita shuffle). Definir como -1 desativa.
* spark.local.dir: Diretório temporário para onde o Spark descarrega dados em disco durante o shuffle.

#### Integração com Armazenamento (S3, Delta, Hive)
Usado para conectar com bancos de dados e Data Lakes.

* spark.sql.extensions: Adiciona extensões de terceiros (ex: io.delta.sql.DeltaSparkSessionExtension).
* spark.sql.catalog.spark_catalog: Define o provedor do catálogo (ex: org.apache.spark.sql.delta.catalog.DeltaCatalog).
* spark.hadoop.fs.s3a.access.key: Chave de acesso para o AWS S3.
* spark.hadoop.fs.s3a.secret.key: Chave secreta para o AWS S3.
* spark.sql.warehouse.dir: Diretório padrão para tabelas gerenciadas pelo Hive/Spark.

#### Serialização e Rede
Ajustes para otimizar a velocidade de envio de dados pela rede.

* spark.serializer: Classe de serialização. Usar org.apache.spark.serializer.KryoSerializer costuma ser muito mais rápido que o padrão.
* spark.kryoserializer.buffer.max: Tamanho máximo do buffer Kryo (ex: 512m).
* spark.network.timeout: Tempo limite padrão para todas as comunicações de rede

In [33]:
spark = SparkSession\
        .builder\
        .config("spark.app.name", "NovaSessao")\
        .config("spark.executor.memory", "1g")\
        .config("spark.executor.cores", "4")\
        .config("spark.cores.max", "4")\
        .config("spark.sql.shuffle.partitions", "10")\
        .getOrCreate()

In [34]:
spark

In [35]:
spark.stop()

Cluster x Executor

```
+-----------------------------------------------------------------------------+
|                                  CLUSTER                                    |
|                                                                             |
|  +-----------------------------------+                                      |
|  |             NÓ MASTER             | <---+ (Gerencia o cluster)           |
|  |  [ Processo Driver: O Gerente ]   |     |                                |
|  +-----------------------------------+     |                                |
|                    |                       |                                |
|     (Envia tarefas | e divide os dados)    |                                |
|                    v                       v                                |
|  +-----------------------------------+   +-------------------------------+  |
|  |           NÓ WORKER 1             |   |          NÓ WORKER 2          |  |
|  |        (Máquina Virtual)          |   |       (Máquina Virtual)       |  |
|  |                                   |   |                               |  |
|  |  +-----------------------------+  |   |  +-------------------------+  |  |
|  |  |         EXECUTOR A          |  |   |  |       EXECUTOR B        |  |
|  |  |    (Processo de Software)   |  |   |  |  (Processo de Software) |  |
|  |  |                             |  |   |  |                         |  |
|  |  |  [Core 1] [Core 2] [Core 3] |  |   |  |  [Core 1] [Core 2]      |  |
|  |  |   (Processam 3 tasks em     |  |   |  |   (Processam 2 tasks em |  |
|  |  |        paralelo)            |  |   |  |        paralelo)        |  |
|  |  +-----------------------------+  |   |  +-------------------------+  |  |
|  +-----------------------------------+   +-------------------------------+  |
+-----------------------------------------------------------------------------+
```

### Simulando executores

In [39]:
spark = SparkSession.builder\
        .master("local[3]")\
        .config("spark.app.name", "SimulacaoExecutores")\
        .config("spark.executor.memory", "450m")\
        .config("spark.executor.instances", "3")\
        .config("spark.executor.cores", "3")\
        .config("spark.sql.shuffle.partitions", '10')\
        .getOrCreate()


In [40]:
spark

In [41]:
print(spark.sparkContext.master)

local[3]


> Simulando 3 executores no proprio computados onde cada executor consome 250 mb usando 3 cores da cpu no cluster